# ST1510 CA1

**Name:** Ethan Chuwa Si Xuan  
**Admin No.:** P2508638  
**Class:** DAAA/FT/1B/04


## Title & Research Question

### “How do estimated salaries vary in relation to key job-related factors within the job market?”

## Import Libraries
Import Pandas for data analysis and Matplotlib for creating visualizations

In [38]:
import pandas as pd # import pandas
import matplotlib.pyplot as plt# import matplotlib
import numpy as np # import numpy

## Load Dataset
I load all five datasets provided in the CA1 folder using relative paths.  
The first four files are CSV files, and the fifth file is an Excel file.  
`low_memory=False` is used for the CSV files to ensure accurate datatype detection.

In [ ]:
df1 = pd.read_csv("CA1 Dataset/postings1.csv", low_memory=False)
df2 = pd.read_csv("CA1 Dataset/postings2.csv", low_memory=False)
df3 = pd.read_csv("CA1 Dataset/postings3.csv", low_memory=False)
df4 = pd.read_csv("CA1 Dataset/postings4.csv", low_memory=False)
df5 = pd.read_excel("CA1 Dataset/postings5.xlsx")

## Initial Data Exploration
To understand the structure and quality of all five datasets, I perform a quick exploration loop.  
This loop displays the head, shape, info, missing values, and a random sample for each dataframe.  
This helps identify data types, missing values, and potential cleaning steps.

In [ ]:
# =============================================================================
# Function: explore_df
# Purpose : Provide a quick structured summary of any DataFrame, including:
#           - shape
#           - head preview
#           - column dtypes
#           - missing value summary (only columns with NaNs)
# =============================================================================
def explore_df(name, df):
    print(f"================ {name} ================")
    print(f"Shape: {df.shape}")
    print()  # spacing

    # -------------------------
    # Preview first 3 rows
    # -------------------------
    print("Head (first 3 rows):")
    display(df.head(3))
    print()

    # -------------------------
    # Show data types
    # -------------------------
    print("Column dtypes:")
    print(df.dtypes)
    print()

    # -------------------------
    # Missing values summary
    # Only display columns that actually have NaN values
    # -------------------------
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    if not missing.empty:
        print("Missing values (top 10 columns):")

        # Convert to dataframe + calculate percentage missing
        missing_df = (
            missing
            .to_frame(name="missing_count")
            .assign(missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(2))
        )

        display(missing_df.head(10))
    else:
        print("No missing values.")

    print("\n")  # extra spacing between dataframes


# =============================================================================
# Loop through all uploaded dataframes
# =============================================================================
dfs = {
    "df1": df1,
    "df2": df2,
    "df3": df3,
    "df4": df4,
    "df5": df5
}

for name, df in dfs.items():
    explore_df(name, df)


At first glance, `df1` and `df5` looks the same with `df1` having more "junk" columns, hence we will use `.compare()` to compare the 2 datasets.

In [ ]:
# Take the first 3 columns from df1 and df5
df1_first3 = df1.iloc[:, :3].copy()
df5_first3 = df5.iloc[:, :3].copy()

# Compare the two DataFrames column-by-column and show differences
df1_first3.compare(
    df5_first3,
    result_names=("df1", "df5")  # rename columns in comparison output
)


It is shown that `df5` has corrupted data, hence `df1` would be used from this point onwards.

We will then check if the seemingly unimportant data in `df1` are actually unimportant.

In [ ]:
# Loop through all columns starting from the 4th column (index 3)
for col in df1.columns[3:]:  
    print(f"Column: {col}")
    
    # Show all unique values in the column
    print("Unique values:", df1[col].unique())
    
    # Count how many unique values the column has
    print("Unique count:", df1[col].nunique())
    
    print("-" * 40)  # separator for readability

The output shows that columns like `Easter Egg` and `Unimportant Column` contain only data that are not useful.

## Data Cleaning & Preparation
Before analysis, I clean and prepare the datasets to ensure accuracy.  
This includes handling missing values, fixing data types, removing irrelevant columns, and preparing the data for analysis.  

### **Handling `df5`**

Although `df5` contains similar job-level fields as `df1`, it was found to contain **corrupted or inconsistent data** during inspection. Several key fields did not match expected values, and comparing the first few columns revealed clear discrepancies between `df5` and the other datasets. These inconsistencies could introduce errors during merging and further analysis.

Therefore, `df5` will **not** be used. Instead, `df1` will serve as the main job table, as it provides a more complete and reliable set of job-level attributes without the corruption issues present in `df5`.

Only the first 3 columns of `df1` are kept as they are the only columns that actually contain useful data.


In [ ]:
# -----------------------------------------------------------------------------
# Function: merge_all
# Purpose : Merge multiple DataFrames together using a common key.
# Usage   : merge_all(df1, df2, df3, on="job_id")
# -----------------------------------------------------------------------------
def merge_all(*dataframes, on):
    out_dataframe = dataframes[0]  # start with the first dataframe
    for dataframe in dataframes[1:]:  
        # merge sequentially with the remaining dataframes
        out_dataframe = pd.merge(out_dataframe, dataframe, on=on)
    return out_dataframe


# -----------------------------------------------------------------------------
# Keep only the first 3 columns from df1 (e.g., job_id + 2 other fields)
# -----------------------------------------------------------------------------
df1_first3 = df1.iloc[:, :3].copy()


# -----------------------------------------------------------------------------
# Merge df1_first3 with df2, df3, df4 using "job_id" as the merge key
# -----------------------------------------------------------------------------
dataframe = merge_all(df1_first3, df2, df3, df4, on="job_id")

# Preview merged result
dataframe.head()


### **Cleaning merged dataset**


`.info()` is then used to view all columns.

In [ ]:
dataframe.info()#shows every column

It is shown that there are a total of 123849 rows.

Next, we will remove columns that are deemed unnecessary to my research question.

In [ ]:
dataframe = dataframe.drop(["description","company_id","job_id","original_listed_time","job_posting_url","application_url","expiry","closed_time","skills_desc","posting_domain","sponsored","compensation_type","company_name","location","work_type"], axis=1)
dataframe

| Column Name             | Why It Is Not Useful for the Research Question |
|-------------------------|------------------------------------------------|
| `description`           | Long unstructured text; cannot be compared or quantified for salary analysis. |
| `company_id`            | Only an identifier; provides no job-related attributes affecting salary. |
| `job_id`                | Unique posting ID; does not describe any factor influencing salary variation. |
| `original_listed_time`  | `listed_time` already exists |
| `job_posting_url`       | URL link; contains no analysable or meaningful information for salary. |
| `application_url`       | Application link; irrelevant to job factors or salary variation. |
| `expiry`                | Expiration date of posting; not related to job attributes or salary. |
| `closed_time`           | Time when posting closed; reflects posting lifespan, not salary drivers. |
| `skills_desc`           | Unstructured text; inconsistent and not quantifiable for salary comparison. |
| `posting_domain`        | Website/platform source; does not influence salary levels. |
| `sponsored`             | Indicates if posting was promoted; unrelated to salary determination. |
| `compensation_type`     | Redundant with pay period information; adds no new salary-related insight. |
| `company_name`          | Employer name; not part of job characteristics affecting salary. |
| `location`              | Not meaningful to clean. |


We will then use `to_datetime()` to convert `listed_time` format to a readable time.

In [ ]:
# Convert the 'listed_time' column from milliseconds to datetime format
dataframe['listed_time'] = pd.to_datetime(
    dataframe['listed_time'], 
    unit='ms'   # timestamps are in milliseconds
)

# Display the updated DataFrame
dataframe


We then check how many unique values there are in `currency`.

In [ ]:
dataframe["currency"].unique()

We can see that there are 6 different types of currency.

We then drop Null values inside of `currency` as there is no way to process salary without information on what currency it is in.

In [ ]:
dataframe["currency"] = dataframe["currency"].fillna("USD")
dataframe.head(20)


We then check for more information on `med_salary`, `min_salary` and `max_salary` to understand how we can proceed to process the data.

In [ ]:
dataframe.head()

In [ ]:
# -----------------------------------------------------------------------------
# Identify rows where median salary is missing
# -----------------------------------------------------------------------------
missing_median = dataframe["med_salary"].isna()

# -----------------------------------------------------------------------------
# Categorise different missing-salary situations
# -----------------------------------------------------------------------------

# Case 1: Has min_salary but no max_salary
has_min_only = (
    missing_median &
    dataframe["min_salary"].notna() &
    dataframe["max_salary"].isna()
)

# Case 2: Has max_salary but no min_salary
has_max_only = (
    missing_median &
    dataframe["min_salary"].isna() &
    dataframe["max_salary"].notna()
)

# Case 3: Has both min and max → median can be calculated
has_both_vals = (
    missing_median &
    dataframe["min_salary"].notna() &
    dataframe["max_salary"].notna()
)

# Case 4: Missing all three (min, max, median)
has_none_vals = (
    missing_median &
    dataframe["min_salary"].isna() &
    dataframe["max_salary"].isna()
)

# -----------------------------------------------------------------------------
# Summary of missing median salary patterns
# -----------------------------------------------------------------------------
print("Total rows with missing median salary:", missing_median.sum())
print("- Rows with MIN salary only:", has_min_only.sum())
print("- Rows with MAX salary only:", has_max_only.sum())
print("- Rows with BOTH min and max salaries:", has_both_vals.sum())
print("- Rows with NEITHER min nor max:", has_none_vals.sum())


We can see that `min_salary` always exists together with `max_salary` when `med_salary` doesn't exist and `med_salary` will always exists when the other 2 don't.

Hence we can fill in missing `med_salary` values using `min_salary` and `max_salary`.

In [ ]:
# -----------------------------------------------------------------------------
# Fill missing median salary using the midpoint of min and max salary
# This applies only where med_salary is NaN and both min/max are available.
# -----------------------------------------------------------------------------
dataframe.loc[:, 'med_salary'] = dataframe['med_salary'].fillna(
    (dataframe['min_salary'] + dataframe['max_salary']) / 2
)

# -----------------------------------------------------------------------------
# Check if any missing salary values remain in the dataset
# -----------------------------------------------------------------------------
dataframe.isnull().sum().sort_values(ascending=True)



We can then proceed to safely drop `max_salary` and `min_salary`.

In [ ]:
dataframe=dataframe.drop(["max_salary","min_salary"],axis=1)
dataframe

We will then convert all currency to USD to allow for comparison later on.

In [ ]:
# -----------------------------------------------------------------------------
# Currency Conversion Function
# Converts salaries from various currencies to USD.
# Rates taken from Morningstar via Google (7/11/2025, 21:37)
# -----------------------------------------------------------------------------
def convert_currency(salary, currency):
    conversion_rates = {
        'CAD': 0.73,  # 1 CAD = 0.73 USD
        'BBD': 0.50,  # 1 BBD = 0.50 USD
        'EUR': 1.08,  # 1 EUR = 1.08 USD
        'AUD': 0.66,  # 1 AUD = 0.66 USD
        'GBP': 1.25   # 1 GBP = 1.25 USD
    }

    # If currency exists in the dictionary, convert it; otherwise assume already USD
    if currency in conversion_rates:
        return salary * conversion_rates[currency]
    return salary


# -----------------------------------------------------------------------------
# Apply conversion on 'med_salary' for all non-USD rows
# Using .apply to pass the entire row into the function
# -----------------------------------------------------------------------------
dataframe['med_salary'] = dataframe.apply(
    lambda row: convert_currency(row['med_salary'], row['currency']),
    axis=1
)

# Display updated DataFrame
dataframe


`.unique()` is then used to understand how many types of pay periods there are.

In [ ]:
dataframe["pay_period"].unique()

In [ ]:
dataframe["formatted_work_type"].unique()

Conversion factors were defined to standardise all salaries into yearly amounts. For hourly roles, annual salary is calculated using the job’s work type (e.g., full-time, part-time) mapped to expected hours per week, multiplied across 52 weeks. For weekly, biweekly, monthly and yearly salaries, fixed period-to-year multipliers are applied. This ensures all roles are converted into a comparable annual salary format.

In [ ]:
# -----------------------------------------------------------------------------
# 1. Assign hours per week based on formatted_work_type
#    Using realistic assumptions for each category.
# -----------------------------------------------------------------------------
work_type_hours = {
    'Full-time': 40,
    'Part-time': 20,
    'Contract': 40,
    'Temporary': 30,
    'Internship': 40,
    'Volunteer': 20,
    'Other': 40
}

# Map the formatted_work_type column to hours/week
dataframe['hours_per_week'] = dataframe['formatted_work_type'].map(work_type_hours)

# Default to 40 hours/week if any value is unmapped
dataframe['hours_per_week'] = dataframe['hours_per_week'].fillna(40)


# -----------------------------------------------------------------------------
# 2. Conversion factors for non-hourly payments
#    (HOURLY is excluded because we handle it separately)
# -----------------------------------------------------------------------------
conversion_factors = {
    'YEARLY': 1,
    'WEEKLY': 52,
    'BIWEEKLY': 26,
    'MONTHLY': 12
}


# -----------------------------------------------------------------------------
# 3. Calculate annual salary based on pay_period
#    HOURLY  → med_salary × hours_per_week × 52
#    OTHERS  → med_salary × conversion_factors
# -----------------------------------------------------------------------------

# Prepare column
dataframe['annual_salary'] = np.nan

# HOURLY rows
hourly_mask = dataframe['pay_period'] == 'HOURLY'

dataframe.loc[hourly_mask, 'annual_salary'] = (
    dataframe.loc[hourly_mask, 'med_salary']
    * dataframe.loc[hourly_mask, 'hours_per_week']
    * 52
)

# NON-HOURLY rows
non_hourly_mask = ~hourly_mask

dataframe.loc[non_hourly_mask, 'annual_salary'] = (
    dataframe.loc[non_hourly_mask, 'med_salary']
    * dataframe.loc[non_hourly_mask, 'pay_period'].map(conversion_factors)
)


# -----------------------------------------------------------------------------
# 4. Display important columns to verify correctness
# -----------------------------------------------------------------------------
dataframe[['med_salary', 'pay_period', 'formatted_work_type', 
          'hours_per_week', 'annual_salary']]


We then filter for salaries that are outside of reasonable range to remove non-representative values that could distort analysis.

In [ ]:
# -----------------------------------------------------------------------------
# Filter out unrealistic yearly salaries
# Keep only values between $10,000 and $500,000
# -----------------------------------------------------------------------------
valid_annual = dataframe['annual_salary'].between(10000, 500000)

# Create a clean copy of the filtered dataset
clean_df = dataframe[valid_annual].copy()

# -----------------------------------------------------------------------------
# Print summary of how many rows were removed
# -----------------------------------------------------------------------------
print("Rows before cleaning:", len(dataframe))
print("Rows after cleaning:", len(clean_df))
print("Rows removed:", len(dataframe) - len(clean_df))

# Replace original with cleaned version
dataframe = clean_df

# Display cleaned DataFrame
dataframe


We then check for any more NaN values to figure out which data to process next.

In [ ]:
dataframe.isnull().sum().sort_values(ascending=True)

We then count how many times each value appears in the `remote_allowed` column.

In [ ]:
print(dataframe['remote_allowed'].value_counts(dropna=False))

There is too much NaN values to drop, hence we will choose to fill them with 0 instead.

In [ ]:
dataframe['remote_allowed'] = dataframe['remote_allowed'].fillna(0)

We will also fill up NaN values in `formatted_experience_level` with unknown instead of dropping data can still provide meaningful insights.

In [ ]:
dataframe['formatted_experience_level'] = dataframe['formatted_experience_level'].fillna("Unknown")

In [ ]:
dataframe.isnull().sum().sort_values(ascending=True)

The dataset contains thousands of inconsistent job titles, making direct comparison difficult. To standardise the titles, a custom dictionary (category_map) was created. Each job category (e.g., Engineering, Management, Finance) is linked to a list of keywords commonly found in titles for that category.

When a job title contains any of the keywords, it is assigned to that category.
This allows us to group similar roles together and analyse salary differences across meaningful job categories, which directly supports the research question.

In [ ]:
# -----------------------------------------------------------------------------
# Category Mapping Dictionary
# Purpose:
#   Map job titles to broader job categories based on keyword matching.
#   Each category contains a list of keywords commonly found in job titles.
#
# Usage (later):
#   Use this dictionary to classify each job title into a standardized category.
#   Example:
#       if any(keyword in title_lower for keyword in category_map["Engineering"])
#           → assign category "Engineering"
# -----------------------------------------------------------------------------
category_map = {
    "Engineering": [
        "engineer", "engineering", "technician", "maintenance", "reliability",
        "mechanic", "hvac", "molding", "silicon", "electrical", "infrastructure"
    ],
    
    "Management": [
        "manager", "supervisor", "director", "lead", "leader", "principal",
        "project manager", "program manager", "area leader", "administrator", "coordinator"
    ],
    
    "Finance": [
        "accountant", "accounts", "finance", "financial", "bookkeeper",
        "payroll", "loan", "bank", "advisor", "wealth", "receivable"
    ],
    
    "Healthcare": [
        "nurse", "clinic", "clinical", "health", "medical", "sterilization",
        "nutrition", "technologist", "psychiatrist", "physician"
    ],
    
    "Customer Service / Admin": [
        "customer", "representative", "support", "assistant", "entry", 
        "clerk", "specialist", "reception", "call center", "scheduling",
        "team member", "client service"
    ],
    
    "Sales & Marketing": [
        "sales", "marketing", "communications", "brand",
        "merchandiser", "business development"
    ],
    
    "IT / Technology": [
        "developer", "software", "it", "java", "sap", "data",
        "tech", "technology", "systems"
    ],
    
    "Education": [
        "faculty", "instructor", "teacher", "professor", "educator"
    ],
    
    "Legal / Compliance / Security": [
        "attorney", "lawyer", "legal", "compliance", "counsel",
        "police", "security", "underwriting"
    ],
    
    "Construction / Skilled Trades": [
        "construction", "laborer", "skilled", "electrician", "welder",
        "operator", "superintendent", "hvac", "carpenter"
    ],
    
    "Transportation / Drivers": [
        "driver", "cdl", "delivery", "transport", "logistics",
        "export", "shipping"
    ],
    
    "Science / Research": [
        "scientist", "research"
    ],
    
    "Restaurant / Food Service": [
        "chef", "cook", "restaurant", "server", "kitchen"
    ],
}


In [ ]:
# -----------------------------------------------------------------------------
# Function: map_job_category
# Purpose:
#   Assign a job category to a job title using keyword matching.
#   Returns the first matching category; otherwise returns "Other".
# -----------------------------------------------------------------------------
def map_job_category(title):
    # Convert title to lowercase for case-insensitive matching
    t = str(title).lower()
    
    # Loop through each category and its list of keywords
    for category, keywords in category_map.items():
        for kw in keywords:
            # If keyword appears anywhere in the title → assign category
            if kw in t:
                return category
    
    # If no category matched
    return "Other"


Sorted category names are then stored in `job_category` column.

In [ ]:
dataframe['job_category'] = dataframe['title'].apply(map_job_category)

In [ ]:
dataframe['job_category'].value_counts()

We will use `.reset_index()` to reset the index of the dataframe.

In [ ]:
dataframe = dataframe.reset_index(drop=True)
dataframe

Hence this is our final dataset that we will be using to plot our charts.

## **Plotting of Charts**

### Selection of Relevant Columns
Initially, several columns such as `remote_allowed`, `application_type`, and `listed_time` were considered as potential factors that might influence salary. However, after further exploration and preliminary plotting, these variables did not show meaningful or interpretable relationships with salary. 

For example, `remote_allowed` contains mostly "Not Specified" values, `application_type` reflects the application method rather than job characteristics, and `listed_time` does not vary enough to produce meaningful trends within the given time window.

Since the research question focuses on understanding how salaries vary across key job-related factors (e.g., experience level, work type, job category, and pay period), these columns were retained in the dataset but excluded from further analysis. This ensures relevant features are prioritised without discarding valid data unnecessarily.


### **Chart 1:** Distribution of Annual Salary

This histogram shows how annual salaries are distributed across all job postings in the dataset. By visualising the frequency of salaries within different ranges, we can identify whether the job market is skewed toward lower or higher-paying roles, detect common salary bands, and spot any unusual extremes. This provides an overall understanding of the salary landscape before comparing salary differences across specific job-related factors.

In [ ]:
# -----------------------------------------------------------------------------
# Histogram of Annual Salary (USD)
# Shows the distribution of yearly salaries after cleaning
# -----------------------------------------------------------------------------
plt.figure(figsize=(10, 6))

# Drop NaNs to avoid plotting errors
plt.hist(dataframe['annual_salary'].dropna(), bins=30, edgecolor='black')

plt.title('Distribution of Annual Salary (USD)')
plt.xlabel('Annual Salary (USD)')
plt.ylabel('Number of Jobs')

plt.tight_layout()
plt.show()



### Interpretation

The distribution of annual salaries is heavily right-skewed, with most jobs concentrated between roughly USD 40,000 and USD 120,000. Very high-paying roles become increasingly rare as salary increases, which is typical in labour-market data where only a small proportion of specialised or senior positions reach the upper salary ranges. This confirms that the dataset follows a realistic salary pattern and provides a reliable foundation for comparing how salaries vary across different job-related factors.


### **Chart 2:** Average Annual Salary by Job Category

It is useful to examine the overall salary levels associated with each job category as this highlights how different types of roles compare in terms of pay, allowing us to identify which categories tend to offer higher salaries and which fall on the lower end. By grouping job postings according to their assigned job_category and calculating the average annual salary for each group, we can identify which categories tend to offer higher or lower pay. This provides a clear overview of salary variation across different types of roles and highlights categories that stand out in the job market.

In [ ]:
# -----------------------------------------------------------------------------
# Compute average annual salary per job category
# Sort values so bars appear in ascending order
# -----------------------------------------------------------------------------
avg_cat = (
    dataframe.groupby('job_category')['annual_salary']
    .mean()
    .sort_values()
)

# -----------------------------------------------------------------------------
# Bar chart: Average Annual Salary by Job Category
# -----------------------------------------------------------------------------
plt.figure(figsize=(12, 6))
bars = plt.bar(avg_cat.index, avg_cat.values)

plt.title('Average Annual Salary by Job Category')
plt.xlabel('Job Category')
plt.ylabel('Annual Salary (USD)')
plt.xticks(rotation=45, ha='right')

# -----------------------------------------------------------------------------
# Add labels on top of each bar
# -----------------------------------------------------------------------------
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,  # Center of bar
        height,                             # Place text above bar
        f'{height:,.0f}',                   # e.g., 75,000
        ha='center', va='bottom', fontsize=8
    )

plt.tight_layout()
plt.show()


### Interpretation

The bar chart displays notable differences in average annual salaries across job categories. The highest-paying sectors are Legal / Compliance / Security (~130,898 USD), Management (~109,652 USD), Engineering (~107,674 USD), and IT / Technology (~107,289 USD), all exceeding the 100,000 USD mark. These fields typically require specialised expertise, higher qualifications, or leadership responsibilities, which explains their stronger salary outcomes.

Mid-range salary categories include Science / Research (~104,270 USD), Healthcare (~98,519 USD), Finance (~85,420 USD), and Other (~83,406 USD), with salaries ranging from around 80,000 to 105,000 USD. These roles often demand technical skills but vary in experience or industry demand.

Lower-paying categories are Sales & Marketing (~74,514 USD), Construction / Skilled Trades (~67,782 USD), Education (~64,542 USD), Customer Service / Admin (~63,168 USD), Transportation / Drivers (~55,995 USD), and Restaurant / Food Service (~46,582 USD). These generally require fewer specialised qualifications or involve entry-level or service-oriented tasks.

Overall, the chart shows that job category strongly influences earning potential, with technical, managerial, and specialised fields offering the highest salaries, while service-based and administrative roles tend to offer lower compensation.

### **Chart 3:** Average Annual Salary by Experience Level

To understand how salary changes with seniority, we calculate the average annual salary for each experience level and visualise the results using a line chart. By ordering the levels from Internship to Executive, the plot highlights the progression of earnings as job responsibilities and expertise increase. This helps reveal whether higher experience levels consistently correspond to higher salaries, providing insight into how seniority influences compensation in the job market.

In [ ]:
# -----------------------------------------------------------------------------
# Define preferred ordering of experience levels
# Only some levels may appear in the dataset, so we filter for existing ones.
# -----------------------------------------------------------------------------
exp_order = [
    'Internship',
    'Entry level',
    'Associate',
    'Mid-Senior level',
    'Director',
    'Executive'
]

# Keep only levels actually present in the data
exp_order_existing = [
    lvl for lvl in exp_order
    if lvl in dataframe['formatted_experience_level'].unique()
]

# -----------------------------------------------------------------------------
# Compute mean annual salary per experience level
# Reindex to enforce the correct ordering
# -----------------------------------------------------------------------------
avg_salary_exp = (
    dataframe.groupby('formatted_experience_level')['annual_salary']
    .mean()
    .reindex(exp_order_existing)
)

# -----------------------------------------------------------------------------
# Line chart: Average Annual Salary by Experience Level
# -----------------------------------------------------------------------------
plt.figure(figsize=(10, 6))
plt.plot(avg_salary_exp.index, avg_salary_exp.values, marker='o', linewidth=2)

plt.title('Average Annual Salary by Experience Level')
plt.xlabel('Experience Level')
plt.ylabel('Average Annual Salary (USD)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


### Interpretation

The line chart shows a clear upward trend in average annual salary as experience level increases. Internship and Entry-level roles offer the lowest salaries, while compensation rises steadily through Associate and Mid-Senior levels. A sharp increase is observed at the Director and Executive levels, reflecting the higher responsibility, expertise, and leadership requirements of senior positions. Overall, the pattern demonstrates that experience level is a strong predictor of salary, with more senior roles consistently earning significantly higher pay.


### **Chart 4:** Annual Salary Distribution by Work Type

To compare how salaries vary across different employment arrangements, we visualise the distribution of annual salaries for each work type using a boxplot. This allows us to observe the median pay, overall spread, and presence of outliers within categories such as full-time, part-time, contract, internship, and temporary roles. By comparing these distributions, we can identify which work types generally offer higher or lower salaries and how compensation varies within each category.

In [ ]:
# -----------------------------------------------------------------------------
# Boxplot: Annual Salary by Work Type
# Shows the distribution and spread of salaries for each work type group.
# -----------------------------------------------------------------------------
plt.figure(figsize=(20, 6), dpi=200)

# Create boxplot grouped by formatted_work_type
dataframe.boxplot(column='annual_salary', by='formatted_work_type')

plt.title('Annual Salary by Work Type')
plt.suptitle('')  # Remove the automatic default title
plt.xlabel('Work Type')
plt.ylabel('Annual Salary (USD)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


### Interpretation

The boxplot shows notable salary differences across work types. Full-time and contract roles offer the highest median salaries and display a wider spread, indicating a broad range of compensation across different industries and seniority levels. Temporary and part-time roles generally pay less, with lower medians and tighter distributions. Internship and volunteer positions show the lowest salaries, reflecting their entry-level or non-paid nature. Overall, the visual confirms that work type is a strong factor influencing salary, with more stable and long-term roles tending to offer higher pay.

It is seen that full-time has a lot of outliers. We will explore the possibilities of why this is the case in the following charts.

### **Chart 5:** Distribution of Experience Levels

To understand the composition of the job market, it is useful to examine how frequently different experience levels appear in the dataset. By calculating the proportion of postings for each experience level and visualising them as a pie chart, we can identify which levels are most common and ensure that any salary comparisons are interpreted in the context of this distribution. Only categories representing at least 1% of the data are included to keep the visual clear and meaningful.

In [ ]:
# -----------------------------------------------------------------------------
# Count occurrences of each experience level
# Then convert to percentages and keep only levels ≥ 1%
# -----------------------------------------------------------------------------
exp_counts = dataframe['formatted_experience_level'].value_counts()
exp_percent = (exp_counts / exp_counts.sum()) * 100

# Filter out very small categories (<1%) to avoid clutter
exp_filtered = exp_percent[exp_percent >= 1]

# -----------------------------------------------------------------------------
# Pie chart: Distribution of Experience Levels (≥1%)
# -----------------------------------------------------------------------------
plt.figure(figsize=(8, 6))

# Create pie chart without labels (only percentages shown)
wedges, texts, autotexts = plt.pie(
    exp_filtered.values,
    labels=None,
    autopct='%1.1f%%',
    startangle=140
)

# Add legend on the right
plt.legend(
    wedges,
    exp_filtered.index,
    title="Experience Level",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

plt.title('Distribution of Experience Levels (≥1%)')
plt.tight_layout()
plt.show()


### Interpretation

The pie chart shows that Mid-Senior level roles make up the largest share of the dataset, followed by Entry-level and Unknown experience levels. Associate and Director positions appear less frequently, while Executive and Internship roles account for only a small portion of postings. This distribution indicates that most job listings target mid-level and early-career candidates, with relatively fewer openings at the highest and lowest ends of the experience spectrum. Understanding this breakdown helps contextualise salary comparisons, as some experience levels are far more common than others.

### **Chart 6:** Work Type Distribution for Mid-Senior Level Roles

We analyse the distribution of work types among Mid-Senior level positions as it is seen to be the largest share of composition in the job market. By counting how many Mid-Senior roles fall under categories such as full-time, contract, part-time, or temporary, this visual reveals which employment formats are most common for mid-level professionals. This helps contextualise salary patterns by showing how job structure differs within a single experience level.

In [ ]:
# -----------------------------------------------------------------------------
# Filter dataset to only Mid-Senior level roles
# -----------------------------------------------------------------------------
mid_df = dataframe[dataframe['formatted_experience_level']=="Executive"]

# Count each work type within Mid-Senior roles
mid_counts = mid_df['formatted_work_type'].value_counts()

# -----------------------------------------------------------------------------
# Bar Chart: Work Type Distribution for Mid-Senior Level Roles
# -----------------------------------------------------------------------------
plt.figure(figsize=(8, 5))
bars = plt.bar(mid_counts.index, mid_counts.values)

plt.title('Work Type Distribution for Mid-Senior Level Roles')
plt.xlabel('Work Type')
plt.ylabel('Number of Jobs')
plt.xticks(rotation=45)

# -----------------------------------------------------------------------------
# Add value labels above each bar
# -----------------------------------------------------------------------------
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,  # Center of bar
        height,                             # Place text above bar
        f'{int(height)}',                   # Convert height to integer label
        ha='center', va='bottom', fontsize=9
    )

plt.tight_layout()
plt.show()



### Interpretation

The chart shows that Mid-Senior level roles are overwhelmingly full-time, with more than 10,000 postings—far exceeding all other work types. Contract and part-time positions appear far less frequently, while temporary, internship, and “other” roles make up only a very small portion of mid-level jobs.

This imbalance helps explain the large number of outliers seen in the earlier full-time salary boxplot. Since full-time roles dominate the dataset, especially at mid-senior level where salaries tend to be higher and more variable, the greater volume naturally results in a wider spread of salaries and more extreme values. This reinforces that full-time positions represent the bulk of mid-level opportunities and contribute most to the observed salary variation.

## **Summary and Conclusion**

In [ ]:
# -----------------------------------------------------------------------------
# 6-Plot Dashboard of Salary & Job Characteristics
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
plt.subplots_adjust(hspace=0.4, wspace=0.3)

# =============================================================================
# 1) Histogram – Distribution of Annual Salary
# =============================================================================
axes[0, 0].hist(dataframe['annual_salary'].dropna(), bins=30, edgecolor='black')
axes[0, 0].set_title('Distribution of Annual Salary (USD)')
axes[0, 0].set_xlabel('Annual Salary (USD)')
axes[0, 0].set_ylabel('Number of Jobs')

# =============================================================================
# 2) Bar – Average Annual Salary by Job Category
# =============================================================================
avg_cat = dataframe.groupby('job_category')['annual_salary'].mean().sort_values()

axes[0, 1].bar(avg_cat.index, avg_cat.values)
axes[0, 1].set_title('Average Annual Salary by Job Category')
axes[0, 1].set_xlabel('Job Category')
axes[0, 1].set_ylabel('Annual Salary (USD)')

# Fix tick spacing, alignment, and margin
axes[0, 1].tick_params(axis='x', rotation=45)
for label in axes[0, 1].get_xticklabels():
    label.set_horizontalalignment('right')
axes[0, 1].margins(x=0.02)

# =============================================================================
# 3) Line – Average Annual Salary by Experience Level
# =============================================================================
exp_order = [
    'Internship', 'Entry level', 'Associate',
    'Mid-Senior level', 'Director', 'Executive'
]

# Keep only experience levels that exist in the dataset
exp_order_existing = [
    lvl for lvl in exp_order
    if lvl in dataframe['formatted_experience_level'].unique()
]

# Compute average salary in correct order
avg_salary_exp = (
    dataframe.groupby('formatted_experience_level')['annual_salary']
    .mean()
    .reindex(exp_order_existing)
)

axes[0, 2].plot(avg_salary_exp.index, avg_salary_exp.values,
                marker='o', linewidth=2)
axes[0, 2].set_title('Average Annual Salary by Experience Level')
axes[0, 2].set_xlabel('Experience Level')
axes[0, 2].set_ylabel('Annual Salary (USD)')
axes[0, 2].tick_params(axis='x', rotation=45)

# =============================================================================
# 4) Boxplot – Annual Salary by Work Type
# =============================================================================
dataframe.boxplot(
    column='annual_salary',
    by='formatted_work_type',
    ax=axes[1, 0]
)

axes[1, 0].set_title('Annual Salary by Work Type')
axes[1, 0].set_xlabel('Work Type')
axes[1, 0].set_ylabel('Annual Salary (USD)')
axes[1, 0].tick_params(axis='x', rotation=45)
fig.suptitle('')  # Remove default boxplot title

# =============================================================================
# 5) Pie – Distribution of Experience Levels (≥1%)
# =============================================================================
exp_counts = dataframe['formatted_experience_level'].value_counts()
exp_percent = (exp_counts / exp_counts.sum()) * 100
exp_filtered = exp_percent[exp_percent >= 1]   # Keep ≥1%

# Pie chart with labels outside the pie
wedges, texts, autotexts = axes[1, 1].pie(
    exp_filtered.values,
    labels=None,
    autopct='%1.1f%%',
    startangle=140,
    labeldistance=1.15,   # Push labels outside
    pctdistance=0.85      # Push % labels toward edge
)

axes[1, 1].set_title('Distribution of Experience Levels (≥1%)')

# Clean legend layout
axes[1, 1].legend(
    wedges,
    exp_filtered.index,
    title="Experience Level",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    fontsize=8
)

# =============================================================================
# 6) Bar – Work Type Distribution for Mid-Senior Roles
# =============================================================================
mid_df = dataframe[dataframe['formatted_experience_level'] == 'Mid-Senior level']
mid_counts = mid_df['formatted_work_type'].value_counts()

axes[1, 2].bar(mid_counts.index, mid_counts.values)
axes[1, 2].set_title('Work Type Distribution (Mid-Senior Level)')
axes[1, 2].set_xlabel('Work Type')
axes[1, 2].set_ylabel('Number of Jobs')
axes[1, 2].tick_params(axis='x', rotation=45)

# Final layout
plt.tight_layout()
plt.show()



**Summary of All Visualisations**

**1. Distribution of Annual Salary (Histogram)**

This histogram shows that most jobs fall within the **$40,000–$120,000** annual salary range, with frequencies dropping sharply beyond $150,000.
This indicates that **mid-range salaries dominate the job market**, while extremely high-paying roles are rare.



**2. Average Annual Salary by Job Category (Bar Chart)**

Different job categories have noticeably different salary levels:

* **Highest-paid sectors** include *Legal/Compliance/Security*, *Management*, *IT/Technology*, *Engineering*, and *Healthcare*.
* **Lower-paid categories** include *Restaurant/Food Service*, *Transportation/Drivers*, and *Customer Service/Admin*.

This demonstrates that **industry or job category has a strong influence on salary expectations**.



**3. Average Annual Salary by Experience Level (Line Chart)**

The line chart shows a **clear upward trend**:

* Salaries increase consistently from **Internship → Entry → Associate → Mid-Senior → Director → Executive**.
* Executive roles offer nearly **4× higher pay** compared to internships.

Experience level is therefore one of the **strongest predictors of salary**.



**4. Annual Salary by Work Type (Boxplot)**

Boxplots reveal the distribution of salaries within each work type:

* **Full-time roles** have the **largest range and highest concentration of high salaries**, including outliers above $200k.
* **Contract and Part-time** roles show lower overall salaries and fewer outliers.
* **Internships and Volunteer** roles have the lowest salaries.

This confirms that **employment type significantly shapes earning potential**.



**5. Distribution of Experience Levels (Pie Chart ≥1%)**

The workforce is dominated by:

* **Mid-Senior level (≈36%)**
* **Entry level (≈25%)**
* **Unknown (≈23%)**

Lower roles like *Director* and *Executive* make up only a small portion of the dataset.
This explains why the overall salary distribution leans toward **mid-range values**—higher-level roles are relatively few.



**6. Work Type Distribution for Mid-Senior Level Positions**

Among Mid-Senior roles:

* **Full-time dominates overwhelmingly (≈10,000 jobs)**.
* Contract and Part-time roles exist but are far less common.

This also explains why **full-time salaries** appear with many outliers in the boxplot—Mid-Senior (higher-paid) individuals are mostly hired full-time.



**Overall Conclusion**

**Research Question:**

**“How do estimated salaries vary in relation to key job-related factors within the job market?”**

**Final Conclusion:**

Across all analyses, it is clear that **estimated salaries vary significantly based on experience level, job category, and work type**.

* **Experience level** is the strongest salary driver, with pay rising steadily from internships to executive positions.
* **Job category** also plays a major role, with specialised and high-responsibility sectors (e.g., Legal, IT, Engineering, Healthcare) offering the highest salaries.
* **Work type** influences salary distribution: full-time roles consistently offer higher earnings, while internships, part-time, and volunteer roles show lower pay ranges.

Overall, salaries do not depend on a single variable. Instead, compensation is determined by the combined influence of experience level, industry category, and employment type. Understanding these patterns helps job seekers and analysts make more informed decisions about salary expectations in the job market.
